# Semana 10 — Da EC2 Pública à Arquitetura de Rede
## Leitura da VPC existente e rastreamento do tráfego

Na Semana 07, a Fábrica Virtual Smart N1 foi implantada em uma EC2 com Docker Compose. Agora investigaremos a infraestrutura que tornou esse acesso possível. A instância não está “diretamente na internet”: ela participa de uma VPC, ocupa uma sub-rede, segue rotas e recebe tráfego permitido por regras.

> **Questão central:** por quais decisões de rede uma requisição passa antes de alcançar o contêiner?

### Resultados de aprendizagem

- localizar CIDR, sub-rede, tabela de rotas, Internet Gateway e Security Group;
- explicar o que torna uma sub-rede pública;
- separar roteamento, endereço público e autorização de porta;
- rastrear entrada e resposta entre navegador e contêiner;
- interpretar CIDR e verificar pertencimento de endereços;
- diagnosticar falhas por camada;
- relacionar os componentes AWS a equivalentes em Azure, GCP e Oracle Cloud.

![Fluxo de tráfego em uma VPC pública](images/semana_10_fluxo_vpc_publica.png)

## 1. O que já existe na implantação

A prática anterior utilizou:

- uma região e uma Zona de Disponibilidade;
- uma VPC fornecida pelo laboratório;
- uma sub-rede associada à instância;
- endereço privado e, durante a sessão, endereço público;
- Security Group liberando portas específicas;
- Docker e Docker Compose dentro da EC2;
- serviços publicados por mapeamento de portas.

Esta aula não recria a EC2. Ela lê e explica a arquitetura existente.

## 2. Camadas que não devem ser confundidas

| Camada | Pergunta | Exemplo |
|---|---|---|
| Endereçamento | Qual é a rede e qual é o host? | `10.0.1.25/24` |
| Roteamento | Para onde enviar o pacote? | `0.0.0.0/0 → igw-...` |
| Acesso | Esta origem pode usar esta porta? | TCP 1880 a partir de um CIDR |
| Sistema operacional | O serviço escuta na porta? | `ss -lntp` |
| Contêiner | A porta foi publicada? | `1880:1880` |
| Aplicação | A rota existe e responde? | HTTP `/` ou `/health` |

Abrir uma porta no Security Group não cria uma rota, não inicia um processo e não publica uma porta Docker.

## 3. CIDR: rede como conjunto de endereços

Em `10.0.1.0/24`, os primeiros 24 bits identificam a rede. Em `10.0.0.0/16`, há um espaço maior que contém várias sub-redes `/24`.

A AWS reserva cinco endereços em cada sub-rede IPv4. Portanto, a quantidade de endereços utilizáveis não é simplesmente `2^(32-prefixo)`.

In [ ]:
from ipaddress import ip_network, ip_address

vpc = ip_network("10.0.0.0/16")
subnet = ip_network("10.0.1.0/24")
instancia = ip_address("10.0.1.25")

print("A sub-rede pertence à VPC?", subnet.subnet_of(vpc))
print("A instância pertence à sub-rede?", instancia in subnet)
print("Total matemático de endereços:", subnet.num_addresses)
print("Estimativa utilizável na AWS:", subnet.num_addresses - 5)

## 4. Tabela de rotas: decisão pelo destino

Cada sub-rede está associada a uma tabela. Uma tabela pública típica contém:

| Destino | Alvo | Significado |
|---|---|---|
| `10.0.0.0/16` | `local` | comunicação interna à VPC |
| `0.0.0.0/0` | Internet Gateway | demais destinos IPv4 |

O roteador escolhe a rota **mais específica**. A rota local `/16` vence a rota padrão `/0` quando o destino está dentro da VPC.

In [ ]:
from ipaddress import ip_network, ip_address

rotas = [
    (ip_network("10.0.0.0/16"), "local"),
    (ip_network("0.0.0.0/0"), "internet-gateway"),
]

def escolher_rota(destino):
    ip = ip_address(destino)
    candidatas = [(rede.prefixlen, alvo, rede) for rede, alvo in rotas if ip in rede]
    return max(candidatas, key=lambda item: item[0])

for destino in ["10.0.2.10", "8.8.8.8"]:
    prefixo, alvo, rede = escolher_rota(destino)
    print(f"{destino:12} -> {rede} -> {alvo}")

## 5. O que torna uma sub-rede pública?

Uma sub-rede é considerada pública quando sua tabela possui rota para um Internet Gateway. Para que uma instância IPv4 seja alcançável da internet, ainda são necessários:

1. Internet Gateway anexado à VPC;
2. rota adequada na tabela associada;
3. IPv4 público ou Elastic IP na interface;
4. regra de entrada no Security Group;
5. serviço escutando e porta publicada pelo contêiner;
6. NACL compatível, quando houver customização.

Portanto, “sub-rede pública” descreve a rota da sub-rede; não significa que todo recurso nela esteja automaticamente exposto.

## 6. Security Group: estado e intenção

Security Groups são stateful: quando uma conexão de entrada permitida é estabelecida, o tráfego de resposta correspondente é aceito sem uma regra espelhada de entrada. As regras autorizam; não existe regra explícita de negação.

Exemplo de análise:

| Porta | Origem | Decisão |
|---|---|---|
| 22/SSH | IP administrativo `/32` | acesso restrito |
| 80/HTTP | `0.0.0.0/0` | aplicação pública |
| 1880/Node-RED | rede da instituição | editor restrito |
| 1883/MQTT | internet inteira | risco elevado sem autenticação/TLS |

O princípio é liberar apenas protocolo, porta e origem necessários.

In [ ]:
from ipaddress import ip_network, ip_address

regras = [
    {"porta": 22, "origem": ip_network("203.0.113.27/32"), "nome": "SSH administrativo"},
    {"porta": 80, "origem": ip_network("0.0.0.0/0"), "nome": "HTTP público"},
    {"porta": 1880, "origem": ip_network("198.51.100.0/24"), "nome": "Node-RED institucional"},
]

def permitido(origem, porta):
    ip = ip_address(origem)
    return [r["nome"] for r in regras if r["porta"] == porta and ip in r["origem"]]

for tentativa in [("203.0.113.27", 22), ("198.51.100.50", 1880), ("192.0.2.10", 1880)]:
    print(tentativa, "->", permitido(*tentativa) or "BLOQUEADO")

## 7. Rastreamento de uma requisição

Considere `http://IP_PUBLICO:1880`:

1. o cliente resolve e envia ao endereço público;
2. a borda da AWS encaminha pelo Internet Gateway;
3. o mapeamento alcança a interface privada da EC2;
4. o Security Group avalia protocolo, porta e origem;
5. o sistema operacional recebe TCP/1880;
6. o Docker encaminha a porta publicada ao contêiner;
7. o Node-RED responde;
8. o tráfego de retorno segue o estado da conexão.

Se a página não abre, a investigação acompanha essa cadeia em vez de alterar regras aleatoriamente.

## 8. Diagnóstico orientado por camadas

| Evidência | Interpretação provável |
|---|---|
| EC2 sem IPv4 público | não há destino público direto |
| sub-rede sem rota ao IGW | não é caminho público |
| timeout externo, serviço local responde | rede ou regra de acesso |
| conexão recusada | host alcançado, mas nada escuta na porta |
| `docker ps` sem publicação | porta não saiu do contêiner |
| HTTP 404 | rede funcionou; rota da aplicação não existe |
| HTTP 500 | rede funcionou; erro interno da aplicação |

Essa leitura evita atribuir todo problema à “AWS” ou ao “firewall”.

### Diagnóstico no Terminal (Comandos Práticos de Rede e Docker)

Em ambiente real (EC2 Linux), o diagnóstico de infraestrutura deve seguir comandos CLI específicos por camada:

```bash
# 1. Verificar conectividade pública e resolução DNS
curl -v http://<IP_PUBLICO_EC2>:1880
nc -zvw3 <IP_PUBLICO_EC2> 1880

# 2. Verificar processos escutando no Sistema Operacional da EC2
sudo ss -lntp | grep 1880
# ou: sudo netstat -tulpn | grep 1880

# 3. Inspecionar portas publicadas pelos contêineres Docker
docker ps --format "table {{.Names}}\t{{.Ports}}\t{{.Status}}"

# 4. Consultar regras do Security Group via AWS CLI
aws ec2 describe-security-groups \
    --group-ids sg-0123456789abcdef0 \
    --query 'SecurityGroups[*].IpPermissions'
```


### Estudo de Caso Prático: Resolução de Incidentes de Acesso

Considere o seguinte cenário reportado pela equipe de automação:
> *"Não conseguimos acessar o painel do Node-RED na porta 1880 da nossa instância EC2."*

**Roteiro de Investigação Passo a Passo:**
1. **Teste Externo:** Executar `curl -v http://IP:1880`. Retorna `Connection timed out`? → Suspeita: Security Group ou Tabela de Rotas.
2. **Teste no Host:** Conectar via SSH e rodar `curl http://localhost:1880`. Responde 200 OK? → O serviço está rodando localmente! A causa raiz é estritamente **Rede / Security Group**.
3. **Verificação do Docker:** Rodar `docker ps`. A porta aparece como `0.0.0.0:1880->1880/tcp`? Se for apenas `1880/tcp` sem mapeamento à esquerda, a porta não foi exposta para o host.


## 9. Equivalentes conceituais

| Conceito | AWS | Microsoft Azure | Google Cloud | Oracle Cloud |
|---|---|---|---|---|
| Rede virtual | VPC | Virtual Network | VPC Network | VCN |
| Sub-rede | Subnet | Subnet | Subnet | Subnet |
| Firewall do recurso | Security Group | Network Security Group | VPC Firewall Rule | Network Security Group/Security List |
| Saída/entrada pública | Internet Gateway | Public IP + sistema de rotas | Default Internet Gateway/rotas | Internet Gateway |
| Tabela de rotas | Route Table | Route Table/UDR | Routes | Route Table |

Os nomes e comportamentos específicos variam. A comparação serve para transferir o raciocínio arquitetural, não para afirmar equivalência perfeita.

## 10. Atividade de investigação da arquitetura existente

Produza um inventário da implantação da Semana 07 contendo:

- CIDR da VPC e da sub-rede;
- associação da tabela de rotas;
- rota local e rota padrão;
- endereço privado e público da EC2;
- regras efetivas do Security Group;
- portas publicadas pelo Docker Compose;
- caminho completo de uma requisição até Node-RED ou Flask.

Depois, formule três falhas hipotéticas em camadas diferentes e indique qual evidência distinguiria cada uma. Não altere recursos indiscriminadamente para “ver se funciona”.

## 11. Síntese

- VPC define o domínio de rede; sub-redes segmentam endereços.
- Tabelas de rotas decidem caminhos; Security Groups decidem permissões.
- Uma rota ao Internet Gateway torna a sub-rede pública, mas não expõe sozinha a instância.
- Docker acrescenta outra camada de publicação de portas.
- O diagnóstico eficiente segue o pacote, camada por camada.

## Bibliografia e aprofundamento

- AMAZON WEB SERVICES. **Amazon VPC User Guide**. <https://docs.aws.amazon.com/vpc/latest/userguide/what-is-amazon-vpc.html>.
- AMAZON WEB SERVICES. **Subnet route tables**. <https://docs.aws.amazon.com/vpc/latest/userguide/subnet-route-tables.html>.
- AMAZON WEB SERVICES. **Internet gateways**. <https://docs.aws.amazon.com/vpc/latest/userguide/VPC_Internet_Gateway.html>.
- AMAZON WEB SERVICES. **Control traffic with security groups**. <https://docs.aws.amazon.com/vpc/latest/userguide/vpc-security-groups.html>.
- KUROSE, James; ROSS, Keith. *Redes de Computadores e a Internet*. 8. ed. Bookman, 2021.
- VERAS, Manoel. *Arquitetura de nuvem: Amazon Web Services*. Brasport, 2013.